# qbraid.runtime.qperfect

> **Billing notice:** the QPerfect MIMIQ emulator is currently **free to run** (0 credits). You need a qBraid API key, but no QPerfect account.

[QPerfect](https://qperfect.io)'s **MIMIQ** is a cloud emulator for gate-model circuits. It ships two simulation engines:

- **Statevector**: exact, but memory grows as $2^n$, so it tops out at around 30 qubits.
- **Matrix product states (MPS)**: compresses the state, so it handles **hundreds of qubits**, provided the circuit doesn't build up too much entanglement.

This notebook submits to MIMIQ through the qBraid platform and covers:

1. Connecting and running a first circuit
2. The `runtime_options` that pick and tune the simulation algorithm
3. Running 100 qubits, past where statevector gives up
4. Reproducible sampling with `seed`
5. **Batch execution**: a whole parameter sweep in one job

In [ ]:
%%capture

# "%%capture" hides the install output; comment it out if you need to debug this step.
%pip install 'qbraid[qiskit,visualization]'

## Connect to the device

`QbraidProvider` picks up your API key from `~/.qbraid/qbraidrc` or the `QBRAID_API_KEY`
environment variable. Inside qBraid Lab it is already configured for you.

In [ ]:
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
device = provider.get_device("qperfect:qperfect:sim:mimiq-emulator")

device.metadata()

256 qubits, `ONLINE`, and zero pricing. MIMIQ takes OpenQASM, and qBraid transpiles to it
for you, so you can pass a Qiskit, Cirq, Braket, pyQuil, or pytket circuit straight to `run()`:

In [ ]:
device.supported_run_inputs()

## A first run

A three-qubit GHZ state, written in Qiskit:

In [ ]:
from qiskit import QuantumCircuit


def ghz(num_qubits: int) -> QuantumCircuit:
    circuit = QuantumCircuit(num_qubits)
    circuit.h(0)
    for qubit in range(num_qubits - 1):
        circuit.cx(qubit, qubit + 1)
    circuit.measure_all()
    return circuit


circuit = ghz(3)
circuit.draw()

In [ ]:
job = device.run(circuit, shots=200)
job.result().data.get_counts()

Two peaks, `000` and `111`, as a GHZ state should give. Keys are little-endian: **qubit 0 is
the rightmost bit**.

With no options, MIMIQ uses its `"auto"` algorithm, which picks an engine for the circuit
on its own.

## Runtime options

MIMIQ's settings go in a **`runtime_options` dict**, which qBraid forwards to the emulator
unchanged:

```python
device.run(circuit, shots=200, runtime_options={"algorithm": "mps", "bonddim": 64})
```

The ones you will reach for most:

| Option | What it does |
|---|---|
| `algorithm` | `"auto"` (default for one circuit), `"statevector"`, or `"mps"` (default for a batch) |
| `bonddim` | MPS bond dimension: the most entanglement kept across any cut. Higher is more accurate and slower |
| `entdim` | MPS entanglement dimension, which bounds the multi-qubit gates applied in one step |
| `mpscutoff` | Singular-value truncation cutoff for MPS. Smaller is more accurate and slower |
| `seed` | Seed for sampling, for reproducible counts |
| `timelimit` | Maximum execution time, in minutes |

MIMIQ also accepts more advanced options (`fuse`, `remove_swaps`, `reorderqubits`,
`mpsmethod`, `mpotraversal`, ...). An option name it doesn't recognise fails the job
rather than being silently ignored.

## Past the statevector limit

Statevector memory doubles with every qubit. Ask for it on 40 qubits and the job fails. The
reason is on the job's status, in `status_message`, which is the first place to look when a
job fails:

In [ ]:
sv_job = device.run(ghz(40), shots=100, runtime_options={"algorithm": "statevector"})
sv_job.wait_for_final_state()

status = sv_job.status()
print(status)
print(status.status_message)

Now go much bigger: a **100-qubit** GHZ state with the default `"auto"` algorithm.

In [ ]:
big_job = device.run(ghz(100), shots=200)
big_counts = big_job.result().data.get_counts()

{f"{key[:8]}…{key[-8:]}": count for key, count in big_counts.items()}

All 100 qubits agree on every shot. That works because a GHZ state, however wide, is only
lightly entangled: any cut through the chain carries one bit of entanglement. `"auto"` sees
that and runs MPS, whose cost depends on entanglement, not on qubit count.

The opposite also holds: a deep, highly entangling circuit can be hard for MPS even at
modest width. That's when you trade accuracy for speed with `bonddim` and `mpscutoff`.

In [ ]:
mps_job = device.run(
    ghz(100),
    shots=200,
    runtime_options={"algorithm": "mps", "bonddim": 32, "timelimit": 5},
)
mps_counts = mps_job.result().data.get_counts()

{f"{key[:8]}…{key[-8:]}": count for key, count in mps_counts.items()}

## Reproducible results with `seed`

Sampling is random, so two identical runs normally give different counts. Fix `seed` and
they match exactly. This is useful for tests, and for comparing other options while holding
the sampling fixed:

In [ ]:
seeded = [
    device.run(ghz(3), shots=200, runtime_options={"seed": 7}).result().data.get_counts()
    for _ in range(2)
]

print(seeded[0])
print(seeded[1])
print("identical:", seeded[0] == seeded[1])

## Batch execution

The MIMIQ device supports **batch jobs**: pass a list of circuits with `as_batch=True` and
they run as **one job**, with one job ID and one status to poll, and you get one set of
counts per circuit, in order.

A natural use is a parameter sweep. Rotate one qubit by $\theta$ with `RY` and the
probability of measuring 1 is $\sin^2(\theta/2)$. Sweep twelve angles in a single job:

In [ ]:
import numpy as np

device.profile.batch_job_support

In [ ]:
angles = np.linspace(0, 2 * np.pi, 12)

sweep = []
for theta in angles:
    circuit = QuantumCircuit(1)
    circuit.ry(theta, 0)
    circuit.measure_all()
    sweep.append(circuit)

batch_job = device.run(sweep, shots=500, as_batch=True)
batch_counts = batch_job.result().data.get_counts()

print(batch_job.id)
print(len(batch_counts), "count dictionaries")

In [ ]:
import matplotlib.pyplot as plt

shots = 500
measured = [counts.get("1", 0) / shots for counts in batch_counts]

fine = np.linspace(0, 2 * np.pi, 200)
plt.plot(fine, np.sin(fine / 2) ** 2, label=r"$\sin^2(\theta/2)$")
plt.plot(angles, measured, "o", label="MIMIQ, one batch job")
plt.xlabel(r"$\theta$")
plt.ylabel("P(1)")
plt.legend()
plt.show()

Twelve circuits, one job. A batch defaults to `algorithm="mps"`, because MIMIQ doesn't
accept `"auto"` for more than one circuit. Pass `runtime_options={"algorithm": "statevector"}`
to simulate every circuit in the batch exactly, which is fine at this size.

Without `as_batch=True`, a list of circuits is submitted as **separate jobs**, one per
circuit, and `run()` returns a list of jobs.

## Summary

- MIMIQ is reached through `QbraidProvider` at `qperfect:qperfect:sim:mimiq-emulator`, free of charge.
- Settings go in `runtime_options`: `algorithm`, `bonddim`, `entdim`, `mpscutoff`, `seed`, `timelimit`, and more.
- Statevector is exact but runs out of memory around 30 qubits; MPS handles hundreds of qubits when entanglement stays low, and `"auto"` chooses for you.
- `as_batch=True` runs many circuits as one job, which suits parameter sweeps.
- When a job fails, `job.status().status_message` says why.